In [ ]:
# W8D2 - RAG Evaluation
# Cell 1: Install required packages

!pip install -q \
    langchain==0.3.27 \
    langchain-community==0.3.27 \
    langchain-core==0.3.72 \
    langchain-text-splitters==0.3.9 \
    langchain-chroma==0.2.5 \
    chromadb==1.0.20 \
    langchain-huggingface \
    sentence-transformers==4.1.0 \
    ragas==0.2.15 \
    datasets==3.6.0 \
    transformers

In [ ]:
!pip uninstall -y langgraph langgraph-sdk langgraph-prebuilt

Found existing installation: langgraph 1.2.11
Uninstalling langgraph-1.2.11:
  Successfully uninstalled langgraph-1.2.11
Found existing installation: langgraph-sdk 0.4.4
Uninstalling langgraph-sdk-0.4.4:
  Successfully uninstalled langgraph-sdk-0.4.4
Found existing installation: langgraph-prebuilt 1.1.0
Uninstalling langgraph-prebuilt-1.1.0:
  Successfully uninstalled langgraph-prebuilt-1.1.0


In [ ]:
# W8D2 - Cell 2
# Import test

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline

from transformers import pipeline

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

print("✅ All imports successful!")
print("✅ W8D2 environment is ready!")

✅ All imports successful!
✅ W8D2 environment is ready!


In [ ]:
# W8D2 - Cell 3
# Create knowledge base documents

documents = [
    Document(
        page_content="""
        Python is a high-level, interpreted programming language.
        It is widely used for web development, automation, data science,
        artificial intelligence, machine learning, and scripting.
        Python uses simple and readable syntax.
        """,
        metadata={"topic": "Python"},
    ),

    Document(
        page_content="""
        Machine Learning is a branch of artificial intelligence.
        It allows computers to learn patterns from data and make
        predictions or decisions without being explicitly programmed
        for every individual task.
        Common machine learning types include supervised learning,
        unsupervised learning, and reinforcement learning.
        """,
        metadata={"topic": "Machine Learning"},
    ),

    Document(
        page_content="""
        Retrieval-Augmented Generation, commonly called RAG,
        combines information retrieval with text generation.
        A RAG system first retrieves relevant documents from a
        knowledge base and then provides those documents as context
        to a language model to generate an answer.
        RAG can help reduce unsupported answers by grounding responses
        in retrieved information.
        """,
        metadata={"topic": "RAG"},
    ),

    Document(
        page_content="""
        ChromaDB is a vector database commonly used in AI applications.
        It stores vector embeddings and allows applications to retrieve
        documents based on semantic similarity.
        In a RAG system, ChromaDB can store document embeddings and
        retrieve relevant chunks for a user's question.
        """,
        metadata={"topic": "ChromaDB"},
    ),

    Document(
        page_content="""
        LangChain is a framework for developing applications powered
        by language models.
        It provides components for prompts, document processing,
        retrievers, vector stores, chains, and agents.
        LangChain can be used to connect language models with
        external data sources.
        """,
        metadata={"topic": "LangChain"},
    ),
]

print(f"✅ Created {len(documents)} documents.")

for i, doc in enumerate(documents, start=1):
    print(f"{i}. {doc.metadata['topic']}")

✅ Created 5 documents.
1. Python
2. Machine Learning
3. RAG
4. ChromaDB
5. LangChain


In [ ]:
# W8D2 - Cell 4
# Split documents into smaller chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"✅ Original documents: {len(documents)}")
print(f"✅ Total chunks created: {len(chunks)}")

print("\nFirst chunk:")
print(chunks[0].page_content)
print("\nMetadata:")
print(chunks[0].metadata)

✅ Original documents: 5
✅ Total chunks created: 9

First chunk:
Python is a high-level, interpreted programming language.
        It is widely used for web development, automation, data science,
        artificial intelligence, machine learning, and scripting.
        Python uses simple and readable syntax.

Metadata:
{'topic': 'Python'}


In [ ]:
# W8D2 - Cell 5
# Create embeddings and ChromaDB vector store

print("Loading embedding model...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding model loaded!")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="w8d2_rag_collection"
)

print("✅ ChromaDB vector store created!")
print(f"✅ Stored {len(chunks)} document chunks.")

Loading embedding model...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!
✅ ChromaDB vector store created!
✅ Stored 9 document chunks.


In [ ]:
# W8D2 - Cell 6
# Create retriever

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("✅ Retriever created!")
print("✅ Retrieval k = 3")

✅ Retriever created!
✅ Retrieval k = 3


In [ ]:
# W8D2 - Cell 7
# Test document retrieval

question = "What is Retrieval-Augmented Generation?"

retrieved_docs = retriever.invoke(question)

print(f"Question: {question}")
print(f"\n✅ Retrieved {len(retrieved_docs)} chunks:\n")

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"--- Retrieved Chunk {i} ---")
    print(doc.page_content.strip())
    print(f"Topic: {doc.metadata['topic']}")
    print()

Question: What is Retrieval-Augmented Generation?

✅ Retrieved 3 chunks:

--- Retrieved Chunk 1 ---
Retrieval-Augmented Generation, commonly called RAG,
        combines information retrieval with text generation.
        A RAG system first retrieves relevant documents from a
        knowledge base and then provides those documents as context
Topic: RAG

--- Retrieved Chunk 2 ---
to a language model to generate an answer.
        RAG can help reduce unsupported answers by grounding responses
        in retrieved information.
Topic: RAG

--- Retrieved Chunk 3 ---
ChromaDB is a vector database commonly used in AI applications.
        It stores vector embeddings and allows applications to retrieve
        documents based on semantic similarity.
        In a RAG system, ChromaDB can store document embeddings and
Topic: ChromaDB



In [ ]:
# W8D2 - Cell 8
# Load local Hugging Face LLM

print("Loading FLAN-T5-base model...")
print("This may take a few minutes on CPU.")

local_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_new_tokens=100
)

llm = HuggingFacePipeline(
    pipeline=local_pipeline
)

print("\n✅ FLAN-T5-base loaded successfully!")

Loading FLAN-T5-base model...
This may take a few minutes on CPU.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu



✅ FLAN-T5-base loaded successfully!


In [ ]:
# W8D2 - Cell 9
# RAG answer generation

def generate_rag_answer(question):
    # Step 1: Retrieve relevant documents
    retrieved_docs = retriever.invoke(question)

    # Step 2: Combine retrieved documents into context
    context = "\n\n".join(
        doc.page_content.strip()
        for doc in retrieved_docs
    )

    # Step 3: Create prompt
    prompt = f"""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{question}

Give a short and direct answer.

Answer:
"""

    # Step 4: Generate answer using local model
    answer = llm.invoke(prompt)

    # Step 5: Store retrieved contexts
    contexts = [
        doc.page_content.strip()
        for doc in retrieved_docs
    ]

    return answer, contexts


print("✅ RAG function created!")

✅ RAG function created!


In [ ]:
# W8D2 - Cell 10
# Test complete RAG pipeline

question = "What is Retrieval-Augmented Generation?"

answer, contexts = generate_rag_answer(question)

print("=" * 60)
print("QUESTION")
print("=" * 60)
print(question)

print("\n" + "=" * 60)
print("GENERATED ANSWER")
print("=" * 60)
print(answer)

print("\n" + "=" * 60)
print("RETRIEVED CONTEXTS")
print("=" * 60)

for i, context in enumerate(contexts, start=1):
    print(f"\nContext {i}:")
    print(context)

print("\n✅ RAG pipeline test completed!")

QUESTION
What is Retrieval-Augmented Generation?

GENERATED ANSWER
combines information retrieval with text generation

RETRIEVED CONTEXTS

Context 1:
Retrieval-Augmented Generation, commonly called RAG,
        combines information retrieval with text generation.
        A RAG system first retrieves relevant documents from a
        knowledge base and then provides those documents as context

Context 2:
to a language model to generate an answer.
        RAG can help reduce unsupported answers by grounding responses
        in retrieved information.

Context 3:
ChromaDB is a vector database commonly used in AI applications.
        It stores vector embeddings and allows applications to retrieve
        documents based on semantic similarity.
        In a RAG system, ChromaDB can store document embeddings and

✅ RAG pipeline test completed!


In [ ]:
# W8D2 - Cell 11
# Create 10 evaluation questions and ground-truth answers

qa_pairs = [
    {
        "question": "What is Python?",
        "ground_truth": (
            "Python is a high-level, interpreted programming language "
            "used for web development, automation, data science, "
            "artificial intelligence, machine learning, and scripting."
        ),
    },
    {
        "question": "What is machine learning?",
        "ground_truth": (
            "Machine Learning is a branch of artificial intelligence "
            "that allows computers to learn patterns from data and "
            "make predictions or decisions."
        ),
    },
    {
        "question": "What are the common types of machine learning?",
        "ground_truth": (
            "The common types of machine learning are supervised learning, "
            "unsupervised learning, and reinforcement learning."
        ),
    },
    {
        "question": "What does RAG stand for?",
        "ground_truth": (
            "RAG stands for Retrieval-Augmented Generation."
        ),
    },
    {
        "question": "What is Retrieval-Augmented Generation?",
        "ground_truth": (
            "RAG combines information retrieval with text generation. "
            "It retrieves relevant documents and provides them as context "
            "to a language model to generate an answer."
        ),
    },
    {
        "question": "Why is RAG useful?",
        "ground_truth": (
            "RAG can help reduce unsupported answers by grounding "
            "responses in retrieved information."
        ),
    },
    {
        "question": "What is ChromaDB?",
        "ground_truth": (
            "ChromaDB is a vector database used to store vector "
            "embeddings and retrieve documents based on semantic similarity."
        ),
    },
    {
        "question": "How is ChromaDB used in RAG?",
        "ground_truth": (
            "ChromaDB can store document embeddings and retrieve "
            "relevant chunks for a user's question in a RAG system."
        ),
    },
    {
        "question": "What is LangChain?",
        "ground_truth": (
            "LangChain is a framework for developing applications "
            "powered by language models."
        ),
    },
    {
        "question": "What components does LangChain provide?",
        "ground_truth": (
            "LangChain provides components for prompts, document "
            "processing, retrievers, vector stores, chains, and agents."
        ),
    },
]

print(f"✅ Created {len(qa_pairs)} Q&A pairs.")

for i, qa in enumerate(qa_pairs, start=1):
    print(f"{i}. {qa['question']}")

✅ Created 10 Q&A pairs.
1. What is Python?
2. What is machine learning?
3. What are the common types of machine learning?
4. What does RAG stand for?
5. What is Retrieval-Augmented Generation?
6. Why is RAG useful?
7. What is ChromaDB?
8. How is ChromaDB used in RAG?
9. What is LangChain?
10. What components does LangChain provide?


In [ ]:
# W8D2 - Cell 12
# Generate answers for all 10 questions

evaluation_data = []

print("=" * 60)
print("GENERATING RAG ANSWERS")
print("=" * 60)

for i, qa in enumerate(qa_pairs, start=1):

    question = qa["question"]
    ground_truth = qa["ground_truth"]

    # Generate answer using our RAG pipeline
    answer, contexts = generate_rag_answer(question)

    evaluation_data.append({
        "question": question,
        "answer": answer,
        "contexts": contexts,
        "ground_truth": ground_truth,
    })

    print(f"\nQ{i}: {question}")
    print(f"Answer: {answer}")
    print(f"Contexts retrieved: {len(contexts)}")

print("\n" + "=" * 60)
print("✅ ALL 10 QUESTIONS PROCESSED")
print("=" * 60)

GENERATING RAG ANSWERS

Q1: What is Python?
Answer: a high-level, interpreted programming language
Contexts retrieved: 3

Q2: What is machine learning?
Answer: a branch of artificial intelligence
Contexts retrieved: 3

Q3: What are the common types of machine learning?
Answer: supervised learning, unsupervised learning, and reinforcement learning
Contexts retrieved: 3

Q4: What does RAG stand for?
Answer: Retrieval-Augmented Generation
Contexts retrieved: 3

Q5: What is Retrieval-Augmented Generation?
Answer: combines information retrieval with text generation
Contexts retrieved: 3

Q6: Why is RAG useful?
Answer: RAG can help reduce unsupported answers by grounding responses in retrieved information
Contexts retrieved: 3

Q7: What is ChromaDB?
Answer: a vector database commonly used in AI applications
Contexts retrieved: 3

Q8: How is ChromaDB used in RAG?
Answer: In a RAG system, ChromaDB can store document embeddings and Retrieval-Augmented Generation, commonly called RAG, combines i

In [ ]:
# W8D2 - Cell 13
# Prepare dataset for Ragas evaluation

from datasets import Dataset

ragas_dataset = Dataset.from_list(evaluation_data)

print("✅ Ragas dataset created!")
print(f"Number of evaluation samples: {len(ragas_dataset)}")

print("\nDataset columns:")
print(ragas_dataset.column_names)

print("\nFirst evaluation sample:")
print(ragas_dataset[0])

✅ Ragas dataset created!
Number of evaluation samples: 10

Dataset columns:
['question', 'answer', 'contexts', 'ground_truth']

First evaluation sample:
{'question': 'What is Python?', 'answer': 'a high-level, interpreted programming language', 'contexts': ['Python is a high-level, interpreted programming language.\n        It is widely used for web development, automation, data science,\n        artificial intelligence, machine learning, and scripting.\n        Python uses simple and readable syntax.', 'Machine Learning is a branch of artificial intelligence.\n        It allows computers to learn patterns from data and make\n        predictions or decisions without being explicitly programmed\n        for every individual task.', 'LangChain is a framework for developing applications powered\n        by language models.\n        It provides components for prompts, document processing,\n        retrievers, vector stores, chains, and agents.\n        LangChain can be used to connect lang

In [ ]:
# W8D2 - Cell 14
# Configure local evaluator for Ragas

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Wrap our local FLAN-T5 model
evaluator_llm = LangchainLLMWrapper(llm)

# Wrap our local Hugging Face embeddings
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

print("✅ Ragas evaluator LLM configured!")
print("✅ Ragas evaluator embeddings configured!")

✅ Ragas evaluator LLM configured!
✅ Ragas evaluator embeddings configured!


In [ ]:
# W8D2 - Cell 15
# Run Ragas evaluation

metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
]

print("=" * 60)
print("STARTING RAGAS EVALUATION")
print("=" * 60)

result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)

print("\n" + "=" * 60)
print("✅ RAGAS EVALUATION COMPLETED")
print("=" * 60)

print(result)

STARTING RAGAS EVALUATION


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1183 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1217 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1127 > 512). Running this sequence through the model will result in indexing errors
ERROR:ragas.executor:Exception raised in Job[0]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[2]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[3]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[4]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[6]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[7]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[10]: TimeoutError()
ER


✅ RAGAS EVALUATION COMPLETED
{'faithfulness': nan, 'answer_relevancy': nan, 'context_precision': nan, 'context_recall': nan}


In [ ]:
# W8D2 - Cell 16
# Display Ragas evaluation scores

print("=" * 60)
print("RAGAS EVALUATION SCORES")
print("=" * 60)

scores = result.to_pandas()

print(scores)

print("\n" + "=" * 60)
print("AVERAGE SCORES")
print("=" * 60)

metric_names = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
]

for metric in metric_names:
    if metric in scores.columns:
        print(f"{metric:20s}: {scores[metric].mean():.4f}")

RAGAS EVALUATION SCORES
                                       user_input  \
0                                 What is Python?   
1                       What is machine learning?   
2  What are the common types of machine learning?   
3                        What does RAG stand for?   
4         What is Retrieval-Augmented Generation?   
5                              Why is RAG useful?   
6                               What is ChromaDB?   
7                    How is ChromaDB used in RAG?   
8                              What is LangChain?   
9         What components does LangChain provide?   

                                  retrieved_contexts  \
0  [Python is a high-level, interpreted programmi...   
1  [Machine Learning is a branch of artificial in...   
2  [for every individual task.\n        Common ma...   
3  [Retrieval-Augmented Generation, commonly call...   
4  [Retrieval-Augmented Generation, commonly call...   
5  [to a language model to generate an answer.\n ...   


In [ ]:
# W8D2 - Cell 17
# Inspect Ragas evaluation result

print("Result type:", type(result))

print("\nResult:")
print(result)

print("\nResult dataframe:")
scores = result.to_pandas()
display(scores)

print("\nMissing values:")
print(scores.isna().sum())
# W8D2 - Cell 18
# Check generated answers

for i, item in enumerate(evaluation_data, start=1):
    print("=" * 60)
    print(f"Q{i}: {item['question']}")
    print("ANSWER:", item["answer"])
    print("CONTEXTS:", len(item["contexts"]))

Result type: <class 'ragas.dataset_schema.EvaluationResult'>

Result:
{'faithfulness': nan, 'answer_relevancy': nan, 'context_precision': nan, 'context_recall': nan}

Result dataframe:


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is Python?,"[Python is a high-level, interpreted programmi...","a high-level, interpreted programming language","Python is a high-level, interpreted programmin...",NaN,NaN,NaN,NaN
1,What is machine learning?,[Machine Learning is a branch of artificial in...,a branch of artificial intelligence,Machine Learning is a branch of artificial int...,NaN,NaN,NaN,NaN
2,What are the common types of machine learning?,[for every individual task.\n Common ma...,"supervised learning, unsupervised learning, an...",The common types of machine learning are super...,NaN,NaN,NaN,NaN
3,What does RAG stand for?,"[Retrieval-Augmented Generation, commonly call...",Retrieval-Augmented Generation,RAG stands for Retrieval-Augmented Generation.,NaN,NaN,NaN,NaN
4,What is Retrieval-Augmented Generation?,"[Retrieval-Augmented Generation, commonly call...",combines information retrieval with text gener...,RAG combines information retrieval with text g...,NaN,NaN,NaN,NaN
5,Why is RAG useful?,[to a language model to generate an answer.\n ...,RAG can help reduce unsupported answers by gro...,RAG can help reduce unsupported answers by gro...,NaN,NaN,NaN,NaN
6,What is ChromaDB?,[ChromaDB is a vector database commonly used i...,a vector database commonly used in AI applicat...,ChromaDB is a vector database used to store ve...,NaN,NaN,NaN,NaN
7,How is ChromaDB used in RAG?,[ChromaDB is a vector database commonly used i...,"In a RAG system, ChromaDB can store document e...",ChromaDB can store document embeddings and ret...,NaN,NaN,NaN,NaN
8,What is LangChain?,[LangChain is a framework for developing appli...,a framework for developing applications powere...,LangChain is a framework for developing applic...,NaN,NaN,NaN,NaN
9,What components does LangChain provide?,[LangChain is a framework for developing appli...,"prompts, document processing, retrievers, vect...","LangChain provides components for prompts, doc...",NaN,NaN,NaN,NaN



Missing values:
user_input             0
retrieved_contexts     0
response               0
reference              0
faithfulness          10
answer_relevancy      10
context_precision     10
context_recall        10
dtype: int64
Q1: What is Python?
ANSWER: a high-level, interpreted programming language
CONTEXTS: 3
Q2: What is machine learning?
ANSWER: a branch of artificial intelligence
CONTEXTS: 3
Q3: What are the common types of machine learning?
ANSWER: supervised learning, unsupervised learning, and reinforcement learning
CONTEXTS: 3
Q4: What does RAG stand for?
ANSWER: Retrieval-Augmented Generation
CONTEXTS: 3
Q5: What is Retrieval-Augmented Generation?
ANSWER: combines information retrieval with text generation
CONTEXTS: 3
Q6: Why is RAG useful?
ANSWER: RAG can help reduce unsupported answers by grounding responses in retrieved information
CONTEXTS: 3
Q7: What is ChromaDB?
ANSWER: a vector database commonly used in AI applications
CONTEXTS: 3
Q8: How is ChromaDB used in RAG?
AN

In [ ]:
# W8D2 - Cell 19
# Test the local FLAN-T5 model directly

test_prompt = """
Answer with only one word: YES

Question:
Is Python a programming language?
"""

test_response = llm.invoke(test_prompt)

print("Local LLM response:")
print(test_response)

Local LLM response:
yes


In [ ]:
# W8D2 - Cell 20
# Configure evaluator LLM and embeddings for each metric

# Configure LLM-based metrics
faithfulness.llm = evaluator_llm
answer_relevancy.llm = evaluator_llm

# Configure embedding-based components
answer_relevancy.embeddings = evaluator_embeddings
context_precision.embeddings = evaluator_embeddings
context_recall.embeddings = evaluator_embeddings

print("✅ Ragas metrics configured!")

✅ Ragas metrics configured!


In [ ]:
# W8D2 - Cell 21
# Re-run Ragas evaluation

result = evaluate(
    dataset=ragas_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)

print("\n" + "=" * 60)
print("RAGAS EVALUATION COMPLETED")
print("=" * 60)

scores = result.to_pandas()

display(scores)

print("\nAVERAGE SCORES:")
for metric in [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
]:
    print(f"{metric:20s}: {scores[metric].mean():.4f}")

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[2]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[3]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[7]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[8]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[10]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[11]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[12]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[14]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[15]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[1]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[5]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[9]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[4]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[6]: TimeoutError()
ERROR:ragas.executor:Exception rais


RAGAS EVALUATION COMPLETED


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is Python?,"[Python is a high-level, interpreted programmi...","a high-level, interpreted programming language","Python is a high-level, interpreted programmin...",NaN,NaN,NaN,NaN
1,What is machine learning?,[Machine Learning is a branch of artificial in...,a branch of artificial intelligence,Machine Learning is a branch of artificial int...,NaN,NaN,NaN,NaN
2,What are the common types of machine learning?,[for every individual task.\n Common ma...,"supervised learning, unsupervised learning, an...",The common types of machine learning are super...,NaN,NaN,NaN,NaN
3,What does RAG stand for?,"[Retrieval-Augmented Generation, commonly call...",Retrieval-Augmented Generation,RAG stands for Retrieval-Augmented Generation.,NaN,NaN,NaN,NaN
4,What is Retrieval-Augmented Generation?,"[Retrieval-Augmented Generation, commonly call...",combines information retrieval with text gener...,RAG combines information retrieval with text g...,NaN,NaN,NaN,NaN
5,Why is RAG useful?,[to a language model to generate an answer.\n ...,RAG can help reduce unsupported answers by gro...,RAG can help reduce unsupported answers by gro...,NaN,NaN,NaN,NaN
6,What is ChromaDB?,[ChromaDB is a vector database commonly used i...,a vector database commonly used in AI applicat...,ChromaDB is a vector database used to store ve...,NaN,NaN,NaN,NaN
7,How is ChromaDB used in RAG?,[ChromaDB is a vector database commonly used i...,"In a RAG system, ChromaDB can store document e...",ChromaDB can store document embeddings and ret...,NaN,NaN,NaN,NaN
8,What is LangChain?,[LangChain is a framework for developing appli...,a framework for developing applications powere...,LangChain is a framework for developing applic...,NaN,NaN,NaN,NaN
9,What components does LangChain provide?,[LangChain is a framework for developing appli...,"prompts, document processing, retrievers, vect...","LangChain provides components for prompts, doc...",NaN,NaN,NaN,NaN



AVERAGE SCORES:
faithfulness        : nan
answer_relevancy    : nan
context_precision   : nan
context_recall      : nan


In [ ]:
# W8D2 - Cell 20
# Configure evaluator for each Ragas metric

faithfulness.llm = evaluator_llm
answer_relevancy.llm = evaluator_llm

answer_relevancy.embeddings = evaluator_embeddings
context_precision.embeddings = evaluator_embeddings
context_recall.embeddings = evaluator_embeddings

print("✅ Ragas metrics configured!")

✅ Ragas metrics configured!


In [ ]:
# W8D2 - Cell 21
# Re-run Ragas evaluation

result = evaluate(
    dataset=ragas_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)

print("\n" + "=" * 60)
print("RAGAS EVALUATION COMPLETED")
print("=" * 60)

scores = result.to_pandas()

display(scores)

print("\nAVERAGE SCORES:")
for metric in [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
]:
    print(f"{metric:20s}: {scores[metric].mean():.4f}")

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[2]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[1]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[4]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[6]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[7]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[8]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[10]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[11]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[14]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[15]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[5]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[3]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[12]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[9]: TimeoutError()
ERROR:ragas.executor:Exception rais


RAGAS EVALUATION COMPLETED


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is Python?,"[Python is a high-level, interpreted programmi...","a high-level, interpreted programming language","Python is a high-level, interpreted programmin...",NaN,NaN,NaN,NaN
1,What is machine learning?,[Machine Learning is a branch of artificial in...,a branch of artificial intelligence,Machine Learning is a branch of artificial int...,NaN,NaN,NaN,NaN
2,What are the common types of machine learning?,[for every individual task.\n Common ma...,"supervised learning, unsupervised learning, an...",The common types of machine learning are super...,NaN,NaN,NaN,NaN
3,What does RAG stand for?,"[Retrieval-Augmented Generation, commonly call...",Retrieval-Augmented Generation,RAG stands for Retrieval-Augmented Generation.,NaN,NaN,NaN,NaN
4,What is Retrieval-Augmented Generation?,"[Retrieval-Augmented Generation, commonly call...",combines information retrieval with text gener...,RAG combines information retrieval with text g...,NaN,NaN,NaN,NaN
5,Why is RAG useful?,[to a language model to generate an answer.\n ...,RAG can help reduce unsupported answers by gro...,RAG can help reduce unsupported answers by gro...,NaN,NaN,NaN,NaN
6,What is ChromaDB?,[ChromaDB is a vector database commonly used i...,a vector database commonly used in AI applicat...,ChromaDB is a vector database used to store ve...,NaN,NaN,NaN,NaN
7,How is ChromaDB used in RAG?,[ChromaDB is a vector database commonly used i...,"In a RAG system, ChromaDB can store document e...",ChromaDB can store document embeddings and ret...,NaN,NaN,NaN,NaN
8,What is LangChain?,[LangChain is a framework for developing appli...,a framework for developing applications powere...,LangChain is a framework for developing applic...,NaN,NaN,NaN,NaN
9,What components does LangChain provide?,[LangChain is a framework for developing appli...,"prompts, document processing, retrievers, vect...","LangChain provides components for prompts, doc...",NaN,NaN,NaN,NaN



AVERAGE SCORES:
faithfulness        : nan
answer_relevancy    : nan
context_precision   : nan
context_recall      : nan


In [ ]:
import ragas

print("Ragas version:", ragas.__version__)

Ragas version: 0.2.15


In [ ]:
# W8D2 - Cell 22
# Load a lightweight local evaluator

print("Loading lightweight evaluator...")

evaluator_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    max_new_tokens=64,
    do_sample=False
)

evaluator_llm_raw = HuggingFacePipeline(
    pipeline=evaluator_pipeline
)

evaluator_llm = LangchainLLMWrapper(evaluator_llm_raw)

print("✅ Lightweight Ragas evaluator loaded!")

Loading lightweight evaluator...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


✅ Lightweight Ragas evaluator loaded!


In [ ]:
# W8D2 - Cell 23
# Configure the lightweight evaluator

evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

faithfulness.llm = evaluator_llm
answer_relevancy.llm = evaluator_llm

answer_relevancy.embeddings = evaluator_embeddings
context_precision.embeddings = evaluator_embeddings
context_recall.embeddings = evaluator_embeddings

print("✅ Lightweight evaluator configured for Ragas!")

✅ Lightweight evaluator configured for Ragas!


In [30]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN:
    print("✅ Hugging Face token connected successfully!")
else:
    print("❌ Token not found. Check Colab Secrets.")

✅ Hugging Face token connected successfully!


In [31]:
from huggingface_hub import login

login(token=HF_TOKEN)

print("✅ Hugging Face login successful!")

✅ Hugging Face login successful!


In [32]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

evaluator_endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    huggingfacehub_api_token=HF_TOKEN,
    max_new_tokens=256,
    temperature=0.0,
)

evaluator_llm_raw = ChatHuggingFace(
    llm=evaluator_endpoint
)

print("✅ Hugging Face Ragas evaluator created!")

✅ Hugging Face Ragas evaluator created!


In [34]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("✅ Token loaded!")

✅ Token loaded!


In [35]:
from huggingface_hub import login

login(token=HF_TOKEN)

print("✅ Hugging Face login successful!")

✅ Hugging Face login successful!


In [36]:
test_response = evaluator_llm_raw.invoke(
    "Answer in one short sentence: What is RAG?"
)

print(test_response.content)

RAG stands for Retrieval-Augmented Generation, a technique that combines retrieval-based methods with generative models.


In [37]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Wrap Hugging Face evaluator for Ragas
evaluator_llm = LangchainLLMWrapper(evaluator_llm_raw)

# Use our existing local embedding model
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

# Configure Ragas metrics
faithfulness.llm = evaluator_llm

answer_relevancy.llm = evaluator_llm
answer_relevancy.embeddings = evaluator_embeddings

context_precision.llm = evaluator_llm
context_precision.embeddings = evaluator_embeddings

context_recall.llm = evaluator_llm
context_recall.embeddings = evaluator_embeddings

print("✅ Ragas evaluator configured!")

✅ Ragas evaluator configured!


In [38]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

# Test with only the first Q&A pair
test_dataset = Dataset.from_list([evaluation_data[0]])

test_result = evaluate(
    dataset=test_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    raise_exceptions=True
)

print(test_result)

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

{'faithfulness': 1.0000, 'answer_relevancy': 1.0000, 'context_precision': 1.0000, 'context_recall': 1.0000}


In [40]:
small_dataset = Dataset.from_list(evaluation_data[:3])

small_result = evaluate(
    dataset=small_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    raise_exceptions=True
)

print("\n✅ 3-question Ragas evaluation completed!")
print(small_result)

Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

ClientResponseError: 402, message='Payment Required', url='https://router.huggingface.co/featherless-ai/v1/chat/completions'